# 156 — Prediction Decomposition

How does the model build its prediction step by step? This module tracks the
logit of the predicted token from embedding through each layer, showing
contributions from attention vs MLP, confidence evolution, and alternative
predictions.

## Why This Matters

Composition prediction decomposition measures how components interact across layers to implement complex computations. Understanding composition — how one head's output feeds into another's input — is essential for tracing multi-step algorithms in transformer circuits.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.prediction_decomposition import (
    logit_buildup,
    attn_vs_mlp_logit_share,
    confidence_evolution,
    alternative_predictions,
    embedding_logit_bias,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

## Logit Buildup

Track the cumulative logit of the predicted token from embedding through each layer.

In [ ]:
result = logit_buildup(model, tokens)
print(f"Target token: {result['target_token']}, Final logit: {result['final_logit']:.4f}\n")
for step in result['steps']:
    bar = '#' * int(abs(step['cumulative_logit']) * 5)
    print(f"{step['stage']:12s}: cum={step['cumulative_logit']:+.4f}  delta={step['delta']:+.4f}  {bar}")

## Attention vs MLP Logit Share

For the top-k predicted tokens, how much comes from attention vs MLP?

In [ ]:
result = attn_vs_mlp_logit_share(model, tokens, top_k=5)
for t in result['per_token']:
    print(f"Token {t['token']:3d}: logit={t['logit']:+.4f}  "
          f"embed={t['embed_contribution']:+.4f}  "
          f"attn={t['attn_contribution']:+.4f} ({t['attn_share']:.0%})  "
          f"mlp={t['mlp_contribution']:+.4f} ({t['mlp_share']:.0%})")

## Confidence Evolution

How does prediction confidence change? When does the model "commit"?

In [ ]:
result = confidence_evolution(model, tokens)
print(f"Final: token {result['final_token']}, committed at: {result['commit_stage']}\n")
for p in result['per_layer']:
    bar = '#' * int(p['confidence'] * 50)
    print(f"{p['stage']:12s}: top={p['top_token']:3d}  conf={p['confidence']:.4f}  "
          f"H={p['entropy']:.2f}  {bar}")

## Alternative Predictions

Track how the rankings of the final top-k tokens evolve through layers.

In [ ]:
result = alternative_predictions(model, tokens, top_k=3)
print(f"Final top tokens: {result['final_top_tokens']}\n")
for p in result['per_layer']:
    ranks = {r['token']: r['rank'] for r in p['rankings']}
    print(f"Layer {p['layer']}: ", end='')
    for tok in result['final_top_tokens']:
        print(f"  tok{tok}@rank{ranks[tok]}", end='')
    print()

## Embedding Logit Bias

What would the model predict from the embedding alone, before any layers?

In [ ]:
result = embedding_logit_bias(model, tokens, top_k=5)
print(f"Embed top: {result['embed_top_token']}, Final top: {result['final_top_token']}")
print(f"Embed rank of final winner: {result['embed_rank_of_final']}")
print(f"Embed entropy: {result['embed_entropy']:.2f}\n")
for p in result['embed_predictions']:
    print(f"  token {p['token']:3d}: logit={p['logit']:+.4f}  prob={p['probability']:.4f}")